In [1]:
# Parameters
summary_config = {"run_run_comparison": False, "run_RTP_summary": False, "run_validation": True, "run_network_validation": True, "summary_list": {"RTP-summary-notebook": ["RTP_index", "RTP_congestion", "RTP_topsheet", "RTP_MIC", "RTP_person", "RTP_household", "RTP_access", "RTP_costs", "RTP_walk_bike", "RTP_emissions", "RTP_mode_share", "RTP_freight", "RTP_transit"], "activitysim-validation-notebook": ["work_from_home", "auto_ownership", "telecommute_frequency", "free_parking", "cdap", "intermediate_stop_frequency", "trip_purpose", "trip_destination_choice", "school_location", "work_location", "mandatory_tour_frequency", "mandatory_tour_scheduling", "non_mandatory_tour_frequency", "non_mandatory_tour_destination_choice", "non_mandatory_tour_scheduling", "joint_tour_frequency", "joint_tour_composition", "atwork_subtours_frequency", "atwork_subtours_destination_choice", "atwork_subtours_scheduling", "atwork_subtour_mode", "tour_mode_choice", "trip_mode_choice"], "daysim-validation-notebook": ["telecommute", "time_choice", "tours", "tour_destination", "transit_pass_ownership", "trips", "trip_destination", "workbased_subtour_generation", "workbased_subtour_mode", "work_location", "work_tour_mode", "work_trip_mode"], "network-validation-notebook": ["JBLM", "supplementals", "transit_validation", "traffic_validation", "bike_validation", "link_analysis"], "run-comparison-notebook": ["topsheet", "population", "parking", "vmt", "transit"]}, "p_output_dir": "outputs/summary", "output_folder": "outputs", "survey_folder": "inputs/base_year/survey", "uncloned_folder": "uncloned", "sc_run_name": "current run", "sc_run_path": "../../../../", "survey_directories": {"survey": "../../../../inputs/base_year/survey"}, "comparison_runs_list": {"2050 new transit, old network": "\\\\modelstation3\\c$\\Workspace\\sc_new_2050_transit\\soundcast", "2050 urbansim": "\\\\modelstation2\\c$\\Workspace\\sc_2050_urbansim2_07_30_25"}, "county_map": {"33": "King", "35": "Kitsap", "53": "Pierce", "61": "Snohomish"}, "uc_list": ["@sov_inc1", "@sov_inc2", "@sov_inc3", "@hov2_inc1", "@hov2_inc2", "@hov2_inc3", "@hov3_inc1", "@hov3_inc2", "@hov3_inc3", "@av_sov_inc1", "@av_sov_inc2", "@av_sov_inc3", "@av_hov2_inc1", "@av_hov2_inc2", "@av_hov2_inc3", "@av_hov3_inc1", "@av_hov3_inc2", "@av_hov3_inc3", "@tnc_inc1", "@tnc_inc2", "@tnc_inc3", "@mveh", "@hveh", "@bveh"], "agency_lookup": {"1": "King County Metro", "2": "Pierce Transit", "3": "Community Transit", "4": "Kitsap Transit", "5": "Washington Ferries", "6": "Sound Transit", "7": "Everett Transit"}, "emissions_scenario": "standard", "tot_veh_model_base_year": 3185281, "speed_bins": [-999999.0, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5, 62.5, 67.5, 72.5, 999999.0], "fac_type_lookup": {"0": 0, "1": 4, "2": 4, "3": 5, "4": 5, "5": 5, "6": 3, "7": 5, "8": 0}, "tod_lookup": {"5to9": 5, "9to15": 9, "15to18": 15, "18to20": 18, "20to5": 20}, "summer_list": [87], "special_route_lookup": {"1671": "A-Line Rapid Ride", "1672": "B-Line Rapid Ride", "1673": "C-Line Rapid Ride", "1674": "D-Line Rapid Ride", "1675": "E-Line Rapid Ride", "1677": "H-Line Rapid Ride", "4950": "Central Link", "6995": "Tacoma Link", "6998": "Sounder South", "6999": "Sounder North", "3701": "Swift Blue Line", "3702": "Swift Green Line"}}
input_config = {"debug_skims_and_paths": False, "model_year": "2023", "base_year": "2023", "landuse_inputs": "23_on_23_v3", "network_inputs": "base_year_2023_final", "db_name": "soundcast_inputs_2023.db", "soundcast_inputs_dir": "R:/e2projects_two/SoundCast/Inputs/rtp_2026_2050", "abm_model": "daysim", "run_accessibility_calcs": False, "run_setup_emme_project_folders": False, "run_setup_emme_bank_folders": False, "run_copy_scenario_inputs": False, "run_import_networks": False, "run_skims_and_paths_free_flow": False, "run_skims_and_paths": False, "run_truck_model": False, "run_supplemental_trips": False, "run_daysim": False, "run_summaries": True, "include_av": False, "include_tnc": True, "tnc_av": False, "include_tnc_to_transit": False, "include_knr_to_transit": False, "include_delivery": False, "include_telecommute": True, "run_integrated": False, "should_build_shadow_price": False, "delete_banks": False, "include_tnc_emissions": True, "add_distance_pricing": False, "distance_rate_dict": {"am": 13.5, "md": 8.5, "pm": 13.5, "ev": 8.5, "ni": 8.5}}


In [2]:
import plotly.express as px
import polars as pl
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

In [3]:
hh = util.get_hh_data(summary_config)
person = util.get_person_data(summary_config)
tour = util.get_tour_data(summary_config)
trip = util.get_trip_data(summary_config)
person_day = util.get_person_day_data(summary_config)
parcel_geog = util.get_parcel_geog(input_config, summary_config)
df_parcel = util.get_parcel_landuse_data(summary_config)

In [4]:
# Load parcels_urbansim input
parcel_urbansim = pl.read_csv(
    os.path.join(summary_config['sc_run_path'], 'outputs/landuse/parcels_urbansim.txt'),
    separator=' ',
    infer_schema_length=20000
).select(['parcelid', 'empofc_p', 'emptot_p'])

In [5]:
df_parcel = parcel_geog.join(parcel_urbansim, left_on='ParcelID', right_on='parcelid', how='left')
df_parcel = df_parcel.with_columns([
    (pl.col('empofc_p') / pl.col('emptot_p')).alias('% office jobs')
]).with_columns([
    pl.when(pl.col('% office jobs').is_null()).then(pl.lit('no jobs'))
      .when(pl.col('% office jobs') < 0.2).then(pl.lit('(0, 0.2]'))
      .when(pl.col('% office jobs') < 0.4).then(pl.lit('(0.2, 0.4]'))
      .when(pl.col('% office jobs') < 0.6).then(pl.lit('(0.4, 0.6]'))
      .when(pl.col('% office jobs') < 0.8).then(pl.lit('(0.6, 0.8]'))
      .otherwise(pl.lit('(0.8, 1]'))
      .alias('% office jobs (bins)')
])

In [6]:
df_person_day = person_day.join(person.join(hh, on=['hhno', 'source'], how='left'), on=['hhno', 'pno', 'source'], how='left')
df_person_day = df_person_day.join(
    df_parcel.select(['ParcelID', 'CountyName', 'district_name', '% office jobs (bins)']),
    left_on='pwpcl', right_on='ParcelID', how='left'
)
df_person_day = df_person_day.join(
    df_parcel.select(['ParcelID', 'CountyName', 'district_name']),
    left_on='hhparcel', right_on='ParcelID', how='left', suffix='_hh'
).rename({'CountyName': 'CountyName_work', 'district_name': 'district_name_work', 'CountyName_hh': 'CountyName_hh', 'district_name_hh': 'district_name_hh'})

In [7]:
# Define worker type
df_person_day = df_person_day.with_columns(
    pl.when(pl.col('source') == 'model').then(pl.lit('commuter')).otherwise(pl.lit(None)).alias('worker_type')
).with_columns(
    pl.when((pl.col('source') == 'model') & (pl.col('pwpcl') == pl.col('hhparcel'))).then(pl.lit('wfh'))
      .when((pl.col('source') == 'model') & (pl.col('wkathome') > 3) & (pl.col('pwpcl') != pl.col('hhparcel'))).then(pl.lit('telecommuter'))
      .otherwise(pl.col('worker_type'))
      .alias('worker_type')
)

In [8]:
# format work at home time
df_person_day = df_person_day.with_columns([
    pl.when((pl.col('wkathome') < 10.0) & (pl.col('wkathome') >= 0.0)).then(pl.col('wkathome').floor())
      .when(pl.col('wkathome') < 0.0).then(pl.lit(0.0))
      .otherwise(pl.lit(10.0))
      .alias('wkathome_int')
]).with_columns(
    pl.when(pl.col('wkathome_int') < 10.0).then(pl.col('wkathome_int').cast(pl.Int64).cast(pl.Utf8))
      .otherwise(pl.lit('10+'))
      .alias('wkathome_hour')
)

# person day data for workers
workers = df_person_day.filter(pl.col('pwtyp') != 0)

In [9]:
# tour data
purpose_dict = {1: 'wktours',
                2: 'sctours',
                3: 'estours',
                4: 'pbtours',
                5: 'shtours',
                6: 'mltours',
                7: 'sotours',
                8: 'retours',
                9: 'metours'}

tour = tour.join(df_person_day, on=['hhno', 'pno', 'day', 'source'], how='left')

tour = tour.with_columns(pl.col('pdpurp').replace_strict(purpose_dict, default=None).alias('tour_purpose'))

# Create bins: bins of 2 miles up to 40 miles
max_bin = 40
bin_size = 2
tour = tour.with_columns(
    pl.when((pl.col('tautodist') >= 0) & (pl.col('tautodist') <= max_bin))
      .then(pl.col('tautodist').truediv(bin_size).floor().mul(bin_size).cast(pl.Int64).cast(pl.Utf8))
      .otherwise(None)
      .alias('dist_bins')
)

## worker counts

In [10]:
def summarize_counts(df: pl.DataFrame, dims: list[str], measure: str = 'pdexpfac') -> pl.DataFrame:
    out = (
        df.group_by(dims)
        .agg([
            pl.col(measure).sum().alias('total count'),
            pl.col(measure).count().alias('sample count')
        ])
    )
    group_dims = [d for d in dims if d != 'worker_type' and d != 'pwtyp' and d != 'wktours']
    if len(group_dims) == 0:
        group_dims = ['source']
    return out.with_columns((pl.col('total count') / pl.col('total count').sum().over(group_dims)).alias('percent'))

# worker counts by worker type
df_worker_count = summarize_counts(workers, ['source', 'worker_type'])

fig = px.bar(df_worker_count.sort(['source']).to_pandas(), x="worker_type", y="percent",
             color="source", barmode="group", hover_data=['sample count', 'total count'],
             title="workers by worker type")
fig.update_layout(height=400, width=700, font=dict(size=11),
                  yaxis_tickformat='.2%')
fig.show()

### full/part-time by worker type and % office jobs in work parcel

In [11]:
df_worker_count = summarize_counts(workers, ['source', 'worker_type', 'pwtyp'])

fig = px.bar(df_worker_count.sort(['source']).to_pandas(), x='pwtyp', y="percent",
             facet_col="worker_type", color="source", hover_data=['sample count', 'total count'],
             barmode="group",
             title="Share of full-time/part-time workers by worker type")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(height=400, width=700, font=dict(size=11),
                  yaxis_tickformat='.2%')
fig.show()

In [12]:
df_worker_count = summarize_counts(workers, ['source', '% office jobs (bins)', 'worker_type'])
df_worker_count = df_worker_count.filter(pl.col('% office jobs (bins)') != "no jobs")

fig = px.bar(df_worker_count.sort(['source', '% office jobs (bins)']).to_pandas(), x='worker_type', y="percent",
             facet_col="% office jobs (bins)", color="source", hover_data=['sample count', 'total count'],
             barmode="group",
             title="worker type by % office jobs in work parcel (bins)")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(height=400, width=900, font=dict(size=11))
fig.for_each_yaxis(lambda a: a.update(tickformat=".2%"))
fig.update_xaxes(categoryorder='array', categoryarray=['commuter', 'telecommuter', 'wfh'])
fig.show()

### worker types in counties and districts

- home location

In [13]:
df_worker_count = summarize_counts(workers, ['source', 'CountyName_hh', 'worker_type'])

fig = px.bar(df_worker_count.sort(['CountyName_hh', 'worker_type']).to_pandas(), x='worker_type', y="percent",
             facet_col="CountyName_hh", color="source", hover_data=['sample count', 'total count'],
             barmode="group",
             title="worker type by county of home location")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(height=400, width=700, font=dict(size=11))
fig.for_each_yaxis(lambda a: a.update(tickformat=".2%"))
fig.update_xaxes(categoryorder='array', categoryarray=['commuter', 'telecommuter', 'wfh'])
fig.show()

In [14]:
df_worker_count = summarize_counts(workers, ['source', 'district_name_hh', 'worker_type'])

fig = px.bar(df_worker_count.sort(['source', 'district_name_hh']).to_pandas(), x='worker_type', y="percent",
             facet_col="district_name_hh", facet_col_wrap=5, color="source",
             barmode="group", hover_data=['sample count', 'total count'],
             title="worker type by district of home location")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(height=700, width=900, font=dict(size=11))
fig.for_each_yaxis(lambda a: a.update(tickformat=".2%"))
fig.update_xaxes(categoryorder='array', categoryarray=['commuter', 'telecommuter', 'wfh'])
fig.show()

- work location

In [15]:
df_worker_count = summarize_counts(workers, ['source', 'CountyName_work', 'worker_type'])

fig = px.bar(df_worker_count.sort(['source', 'CountyName_work']).to_pandas(), x='worker_type', y="percent",
             facet_col="CountyName_work", color="source",
             barmode="group", hover_data=['sample count', 'total count'],
             title="worker type by county of work location")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(height=400, width=700, font=dict(size=11))
fig.for_each_yaxis(lambda a: a.update(tickformat=".2%"))
fig.update_xaxes(categoryorder='array', categoryarray=['commuter', 'telecommuter', 'wfh'])
fig.show()

In [16]:
df_worker_count = summarize_counts(workers, ['source', 'district_name_work', 'worker_type'])

fig = px.bar(df_worker_count.sort(['source', 'district_name_work']).to_pandas(), x='worker_type', y="percent",
             facet_col="district_name_work", facet_col_wrap=5, color="source",
             barmode="group", hover_data=['sample count', 'total count'],
             title="worker type by district of work location")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(height=700, width=900, font=dict(size=11))
fig.for_each_yaxis(lambda a: a.update(tickformat=".2%"))
fig.update_xaxes(categoryorder='array', categoryarray=['commuter', 'telecommuter', 'wfh'])
fig.show()

### worker counts by telework hours

In [17]:
# population in each telework hour
df_hour_count = (
    workers.group_by(['source', 'wkathome_int', 'wkathome_hour'])
    .agg(pl.col('pdexpfac').sum().alias('pdexpfac'))
    .with_columns((pl.col('pdexpfac') / pl.col('pdexpfac').sum().over(['source'])).alias('percent'))
)

fig = px.bar(df_hour_count.sort(['source', 'wkathome_int']).to_pandas(), x="wkathome_hour", y="percent", color="source",
             barmode="group",
             title="share of workers by telework hour")
fig.update_layout(height=350, width=700, font=dict(size=11),
                  xaxis=dict(dtick=1),
                  yaxis_tickformat='.2%')
fig.update_xaxes(categoryorder='array', categoryarray=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10+'])
fig.show()

## work tours

In [18]:
_df = summarize_counts(workers.filter(pl.col('worker_type') == "telecommuter"), ['source', 'wktours'])
fig = px.bar(_df.to_pandas(), x="wktours", y="percent", color="source",
             barmode="group",
             title="teleworkers: number of work tours")
fig.update_layout(height=350, width=700, font=dict(size=11),
                  yaxis_tickformat='.2%')
fig.show()

In [19]:
_df = summarize_counts(workers.filter(pl.col('worker_type') == "commuter"), ['source', 'wktours'])

fig = px.bar(_df.to_pandas(), x="wktours", y="percent", color="source",
             barmode="group",
             title="commuters: number of work tours")
fig.update_layout(height=350, width=700, font=dict(size=11),
                  yaxis_tickformat='.2%')
fig.show()

In [20]:
_df = summarize_counts(workers.filter(pl.col('worker_type') == "commuter"), ['source', 'pwtyp', 'wktours'])

fig = px.bar(_df.to_pandas(), x='wktours', y="percent",
             facet_col="pwtyp", color="source",
             barmode="group",
             title="commuter: number of work tours by full-/part-time workers")
fig.update_traces(hovertemplate="share of workers: %{y:.2%}<br>" +
                                "worker counts: %{customdata[0]:.0f}")
fig.update_layout(height=400, width=700, font=dict(size=11),
                  yaxis_tickformat='.2%')
fig.show()

In [21]:
_df = summarize_counts(workers.filter(pl.col('worker_type') == "wfh"), ['source', 'wktours'])

fig = px.bar(_df.to_pandas(), x="wktours", y="percent", color="source",
             barmode="group",
             title="work from home workers: number of work tours")
fig.update_layout(height=350, width=700, font=dict(size=11),
                  yaxis_tickformat='.2%')
fig.show()

In [22]:
_df = summarize_counts(workers.filter(pl.col('worker_type') == "wfh"), ['source', 'pwtyp', 'wktours'])

fig = px.bar(_df.to_pandas(), x='wktours', y="percent",
             facet_col="pwtyp", color="source",
             barmode="group",
             title="work from home: number of work tours by full-/part-time workers")
fig.update_traces(hovertemplate="share of workers: %{y:.2%}<br>" +
                                "worker counts: %{customdata[0]:.0f}")
fig.update_layout(height=400, width=700, font=dict(size=11),
                  yaxis_tickformat='.2%')
fig.show()

In [23]:
_df = summarize_counts(workers.filter(pl.col('wktours') > 0), ['source', 'wkathome_int', 'wkathome_hour'])

_df2 = df_hour_count.select(['source', 'wkathome_int', 'wkathome_hour', 'pdexpfac']).rename({'pdexpfac': 'total_workers'})
_df = _df.join(_df2, on=['source', 'wkathome_int', 'wkathome_hour'], how='left').with_columns(
    (pl.col('total count') / pl.col('total_workers')).alias('percent')
)

fig = px.bar(_df.sort(['source', 'wkathome_int']).to_pandas(), x="wkathome_hour", y="percent", color="source",
             barmode="group",
             title="share of people making 1+ work tours by telework hours")
fig.update_layout(height=300, width=700, font=dict(size=11),
                  xaxis=dict(dtick=1),
                  yaxis_tickformat='.2%')
fig.update_xaxes(categoryorder='array', categoryarray=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10+'])
fig.show()

## Tour rates by destination purpose for each worker type



In [24]:
def calc_rates(df: pl.DataFrame):
    # use person day file to calculate tour rates by purpose

    weighted_cols = ['wbtours', 'uwtours', 'wktours', 'sctours', 'estours', 'pbtours', 'shtours',
                     'mltours', 'sotours', 'retours', 'metours']
    df_w = df.with_columns([(pl.col(col) * pl.col('pdexpfac')).alias(col + '_wt') for col in weighted_cols])

    out_frames = []
    for col in ['wbtours', 'uwtours', 'wktours', 'sctours', 'estours', 'pbtours', 'shtours', 'mltours', 'sotours']:
        num = df_w.group_by(['source', 'worker_type']).agg(pl.col(col + '_wt').sum().alias('num'))
        denom = df_w.group_by(['source', 'worker_type']).agg(pl.col('pdexpfac').sum().alias('denom'))
        rates = num.join(denom, on=['source', 'worker_type'], how='left').with_columns(
            (pl.col('num') / pl.col('denom')).alias('rate')
        ).with_columns(pl.lit(col).alias('purpose')).select(['source', 'worker_type', 'purpose', 'rate'])
        out_frames.append(rates)

    df_tour_rates_long = pl.concat(out_frames, how='vertical')
    return df_tour_rates_long.pivot(values='rate', index=['source', 'purpose'], on='worker_type')

In [25]:
df_tour_rates = calc_rates(workers)

In [26]:
# # tour counts by worker type and tour purpose

def plot_tour_rate(df: pl.DataFrame, worker_type: str, title: str):

    fig = px.bar(df.to_pandas(), x="purpose", y=worker_type, color="source",
                 barmode="group",
                 title=title)
    fig.update_layout(height=300, width=700, font=dict(size=11),
                      yaxis_tickformat='.2f', yaxis_title='Tour Rate')
    fig.show()

In [27]:
df_tour_rates = calc_rates(workers)
plot_tour_rate(df_tour_rates.select(['source', 'commuter', 'purpose']),
               'commuter',
               "Tour Rates by Destination Purpose"
)

In [28]:
workers_no_work_tours = workers.filter(pl.col('wktours') == 0)
df_tour_rates = calc_rates(workers_no_work_tours)
plot_tour_rate(df_tour_rates.select(['source', 'commuter', 'purpose']),
               'commuter',
               "Commuter with no work tours: tour rates by destination purpose"
)

In [29]:
workers_missing_work_loc = workers.filter(pl.col('pwpcl') == -1)
if workers_missing_work_loc.height > 0:
    df_tour_rates = calc_rates(workers_missing_work_loc)
    plot_tour_rate(df_tour_rates.select(['source', 'commuter', 'purpose']),
                   'commuter',
                   "Commuters with missing work location: tour rates by destination purpose"
    )

In [30]:
df_tour_rates = calc_rates(workers)
plot_tour_rate(df_tour_rates.select(['source', 'telecommuter', 'purpose']),
               'telecommuter',
               "Tour rates by destination purpose"
)

In [31]:
df_tour_rates = calc_rates(workers)
plot_tour_rate(df_tour_rates.select(['source', 'wfh', 'purpose']),
               'wfh',
               " tour rates by destination purpose"
)

## Tour distances by purpose

In [32]:
df_tour_distance = (
    tour.group_by(['source', 'worker_type', 'tour_purpose', 'dist_bins'])
    .agg(pl.col('toexpfac').sum().alias('toexpfac'))
    .with_columns(
        (pl.col('toexpfac') / pl.col('toexpfac').sum().over(['tour_purpose', 'worker_type', 'source'])).alias('percent')
    )
)

In [33]:
def plot_tour_distance(df: pl.DataFrame, dpurp: str, worker_type_list: list[str]):

    df_plot = df.filter((pl.col('tour_purpose') == dpurp) & pl.col('worker_type').is_in(worker_type_list))

    fig2 = px.line(df_plot.to_pandas(), x='dist_bins', y="percent", color="worker_type", template="simple_white",
                   facet_col='source',
                   title=dpurp + " tour distance")

    fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig2.update_layout(height=400, width=700, font=dict(size=11),
                       yaxis_tickformat='.2%')
    fig2.show()


plot_tour_distance(df_tour_distance, "wktours", ["wfh", "commuter", "telecommuter"])

In [34]:
plot_tour_distance(df_tour_distance, "sctours", ["wfh", "commuter", "telecommuter"])

In [35]:
plot_tour_distance(df_tour_distance, "estours", ["wfh", "commuter", "telecommuter"])

In [36]:
plot_tour_distance(df_tour_distance, "pbtours", ["wfh", "commuter", "telecommuter"])

In [37]:
plot_tour_distance(df_tour_distance, "shtours", ["wfh", "commuter", "telecommuter"])

In [38]:
plot_tour_distance(df_tour_distance, "mltours", ["wfh", "commuter", "telecommuter"])

In [39]:
plot_tour_distance(df_tour_distance, "sotours", ["wfh", "commuter", "telecommuter"])